<a href="https://colab.research.google.com/github/viethaa/f1-big-data-analytics/blob/main/formula1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align="center"><b>Performance Analysis in Formula One</b></h1>

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Donec risus odio, blandit et sapien in, pharetra convallis nulla. Cras facilisis, arcu vitae sollicitudin ornare, velit arcu lacinia quam, vitae consequat mi felis quis dolor. Donec tincidunt commodo lectus nec porta. Pellentesque eu felis erat. Cras ac porta turpis. Quisque tincidunt auctor mattis. Maecenas convallis orci mauris, tempor rutrum nibh sodales sed. Curabitur in enim a leo tempus rutrum id vitae mauris. Quisque ullamcorper posuere fringilla. Morbi et ante lacinia justo dignissim placerat. The project repository can be found on github: https://github.com/viethaa/f1-big-data-analytics

In [4]:
# Install Required Packages
!pip install fastf1
!pip install pandas matplotlib seaborn plotly

# Import Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import fastf1
import fastf1.plotting
from fastf1.core import Laps

# Enable Cache
os.makedirs('cache', exist_ok=True)
fastf1.Cache.enable_cache('cache')

---


<h1 align="center"><b>Section 1: Tire Compound and Degradation</b></h1>



---

In [ ]:
# Import Libraries
import os
import fastf1
import fastf1.plotting
import pandas as pd
import plotly.graph_objects as go

# Enable Cache
os.makedirs("cache", exist_ok=True)
fastf1.Cache.enable_cache("cache")


# Load Session
session = fastf1.get_session(2024, "Spain", "R")
session.load()
laps = session.laps

# Stint Data
stints = (
    laps.groupby(["Driver", "Stint", "Compound"])
        .agg(LapStart=("LapNumber", "min"),
             LapEnd=("LapNumber", "max"))
        .reset_index()
)

stints["StintLength"] = stints["LapEnd"] - stints["LapStart"] + 1
df = stints.copy()

# Driver Order
results = session.results.sort_values("Position")
driver_order = results["Abbreviation"].tolist()

df["Driver"] = pd.Categorical(
    df["Driver"],
    categories=driver_order,
    ordered=True
)

df = df.sort_values(["Driver", "Stint"])

# Tire Colors
TIRE_COLORS = {
    "SOFT": "#1B4F72",
    "MEDIUM": "#B7950B",
    "HARD": "#922B21",
}

# Team Colors
team_colors = {
    row["Abbreviation"]:
    fastf1.plotting.get_team_color(row["TeamName"], session=session)
    for _, row in session.results.iterrows()
}

fig = go.Figure()
added_to_legend = set()

for _, row in df.iterrows():

    compound = row["Compound"]

    fig.add_trace(
        go.Bar(
            x=[row["StintLength"]],
            y=[row["Driver"]],
            base=row["LapStart"],
            orientation="h",
            width=0.6,

            marker=dict(
                color=TIRE_COLORS.get(compound),
                opacity=0.95,
                line=dict(
                    width=1.5,
                    color="rgba(0,0,0,0.6)"
                )
            ),

            customdata=[[
                compound,
                row["Stint"],
                row["LapStart"],
                row["LapEnd"]
            ]],

            hovertemplate="<b>%{y}</b><br>"
                          "Compound: %{customdata[0]}<br>"
                          "Stint: %{customdata[1]}<br>"
                          "Lap Range: %{customdata[2]} - %{customdata[3]}"
                          "<extra></extra>",

            name=compound,
            showlegend=compound not in added_to_legend
        )
    )

    added_to_legend.add(compound)


# Layout
fig.update_layout(
    title=dict(
        text="Tire Compound Strategy in 2024 Spain Grand Prix",
        x=0.02,
        font=dict(size=30)
    ),

    barmode="overlay",

    plot_bgcolor="#111111",
    paper_bgcolor="#111111",

    font=dict(color="#EAEAEA", family="Courier New"),

    xaxis=dict(
        title=dict(text="Lap Number", font=dict(size=26)),
        tickfont=dict(size=16),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.06)",
        zeroline=False,
        dtick=5
    ),

    yaxis=dict(
        title=dict(text="Driver", font=dict(size=26)),
        tickfont=dict(size=16),
        showgrid=False,
        autorange="reversed",
        tickmode="array",
        tickvals=driver_order,
        ticktext=[
            f"<span style='color:{team_colors[d]}'>{d}</span>"
            for d in driver_order
        ]
    ),

    legend=dict(
        title="Tire Compound:",
        font=dict(size=15),
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        bgcolor="rgba(0,0,0,0)"
    ),

    margin=dict(l=140, r=40, t=100, b=80)
)

fig.show()

core           INFO 	Loading data for Spanish Grand Prix - Race [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Spanish Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
INFO:fastf1.fastf1.req:Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core         

In [ ]:
# Import Libaries
import os
import fastf1
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Enable Cache
os.makedirs("cache", exist_ok=True)
fastf1.Cache.enable_cache("cache")


# Load Session
session = fastf1.get_session(2024, "Spain", "R")
session.load()


# Clean Laps
laps = session.laps.copy()

laps = laps[
    (laps["PitInTime"].isna()) &
    (laps["PitOutTime"].isna())
]

laps = laps.dropna(subset=["LapTime"])
laps = laps[laps["Compound"].isin(["SOFT", "MEDIUM", "HARD"])]

laps["LapTimeSec"] = laps["LapTime"].dt.total_seconds()

laps = laps.sort_values(["Driver", "Stint", "LapNumber"])

laps["TireAge"] = (
    laps.groupby(["Driver", "Stint"])
        .cumcount() + 1
)


# Remove Extreme Slow Laps
upper_limit = laps["LapTimeSec"].quantile(0.95)
laps = laps[laps["LapTimeSec"] < upper_limit]


# Limit Stint Length
laps = laps[laps["TireAge"] <= 30]


# Plot
fig = go.Figure()

TIRE_COLORS = {
    "SOFT": "#1E90FF",
    "MEDIUM": "#FFD700",
    "HARD": "#FF2D2D",
}

for compound in ["SOFT", "MEDIUM", "HARD"]:

    comp_data = laps[laps["Compound"] == compound]

    # If compound barely used, skip
    if len(comp_data) < 5:
        continue

    # Raw Dots
    fig.add_trace(
        go.Scatter(
            x=comp_data["TireAge"],
            y=comp_data["LapTimeSec"],
            mode="markers",
            name=f"{compound} Laps",
            marker=dict(
                color=TIRE_COLORS[compound],
                size=5,
                opacity=0.15
            ),
            legendgroup=compound
        )
    )

    # Average per Tire Age
    avg_data = (
        comp_data.groupby("TireAge")["LapTimeSec"]
        .mean()
        .reset_index()
        .sort_values("TireAge")
    )


    # Compute total change
    start_time = avg_data["LapTimeSec"].iloc[0]
    end_time = avg_data["LapTimeSec"].iloc[-1]
    total_change = end_time - start_time


    # Linear regression slope
    z = np.polyfit(comp_data["TireAge"], comp_data["LapTimeSec"], 1)
    slope = z[0]


    # Average trend line
    fig.add_trace(
        go.Scatter(
            x=avg_data["TireAge"],
            y=avg_data["LapTimeSec"],
            mode="lines",
            name=f"{compound} Trend ({total_change:+.2f}s total, {slope:+.3f}s/lap)",
            line=dict(
                color=TIRE_COLORS[compound],
                width=9
            ),
            legendgroup=compound
        )
    )


# Layout
fig.update_layout(
    title="Tire Degradation in 2024 Spain Grand Prix",
    plot_bgcolor="#1c1c1c",
    paper_bgcolor="#1c1c1c",
    font=dict(color="#F1E9E9", family="Courier New", size=18),


    xaxis=dict(
        title="Tire Age (Laps on Tire)",
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)"
    ),

    yaxis=dict(
        title="Lap Time (Seconds)",
        showgrid=True,
        gridcolor="rgba(255,255,255,0.08)"
    ),

    legend=dict(font=dict(size=16)),
    margin=dict(l=100, r=40, t=100, b=80)
)

fig.show()

core           INFO 	Loading data for Spanish Grand Prix - Race [v3.8.2]
INFO:fastf1.fastf1.core:Loading data for Spanish Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_status_d

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

---


<h1 align="center"><b>Graph 2: Pit Stop Efficiency</b></h1>



---

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

In [6]:
import os
import fastf1
import fastf1.plotting
import pandas as pd
import plotly.graph_objects as go

# --------------------------------------------------
# Cache
# --------------------------------------------------
os.makedirs("cache", exist_ok=True)
fastf1.Cache.enable_cache("cache")

YEAR = 2025
LAPS_BEFORE = 5
LAPS_AFTER = 50

# --------------------------------------------------
# Schedule
# --------------------------------------------------
schedule = fastf1.get_event_schedule(
    YEAR,
    include_testing=False
)

all_stops = []
driver_colors = {}

# --------------------------------------------------
# Load races
# --------------------------------------------------
for _, event in schedule.iterrows():

    race_name = event["EventName"]
    round_num = event["RoundNumber"]

    try:
        session = fastf1.get_session(
            YEAR,
            round_num,
            "R"
        )

        session.load(
            laps=True,
            telemetry=False,
            weather=False,
            messages=False
        )

        laps = session.laps.copy()

        if laps.empty:
            print(f"✗ No lap data: {race_name}")
            continue

        print(f"✓ {race_name}")

    except Exception as e:
        print(f"✗ Skipped {race_name}: {e}")
        continue

    total_laps = int(laps["LapNumber"].max())

    # --------------------------------------------------
    # Driver colors
    # --------------------------------------------------
    for _, row in session.results.iterrows():

        drv = row["Abbreviation"]

        if drv not in driver_colors:

            try:
                driver_colors[drv] = (
                    fastf1.plotting.get_team_color(
                        row["TeamName"],
                        session=session
                    )
                )

            except Exception:
                driver_colors[drv] = "#888888"

    # --------------------------------------------------
    # First pit stop per driver
    # --------------------------------------------------
    first_pits = (
        laps[laps["PitInTime"].notna()]
        .groupby("Driver")["LapNumber"]
        .min()
        .reset_index()
        .rename(columns={"LapNumber": "PitLap"})
    )

    # --------------------------------------------------
    # Build relative lap windows
    # --------------------------------------------------
    for _, pit_row in first_pits.iterrows():

        driver = pit_row["Driver"]
        pit_lap = int(pit_row["PitLap"])

        dlaps = (
            laps[laps["Driver"] == driver]
            .sort_values("LapNumber")
        )

        target_before = max(
            1,
            pit_lap - LAPS_BEFORE
        )

        target_after = min(
            total_laps,
            pit_lap + LAPS_AFTER
        )

        rows = []

        for lap_num in range(
            target_before,
            target_after + 1
        ):

            lap_row = dlaps[
                dlaps["LapNumber"] == lap_num
            ]

            if (
                not lap_row.empty
                and pd.notna(
                    lap_row.iloc[0]["Position"]
                )
            ):

                rows.append({
                    "RelLap": (
                        lap_num - pit_lap
                    ),

                    "Position": int(
                        lap_row.iloc[0]["Position"]
                    )
                })

        # Require enough data points
        if len(rows) >= 3:

            all_stops.append({
                "Driver": driver,
                "Race": race_name,
                "PitLap": pit_lap,

                "df": (
                    pd.DataFrame(rows)
                    .sort_values("RelLap")
                ),

                "Color": driver_colors.get(
                    driver,
                    "#888888"
                )
            })

# --------------------------------------------------
# Summary
# --------------------------------------------------
print(
    f"\nTotal pit stops collected: {len(all_stops)}"
)

# --------------------------------------------------
# Convert to position delta
# --------------------------------------------------
valid_stops = []

for stop in all_stops:

    df = stop["df"].copy()

    pit_row = df[df["RelLap"] == 0]

    if pit_row.empty:
        continue

    pit_pos = int(
        pit_row.iloc[0]["Position"]
    )

    df["Delta"] = (
        pit_pos - df["Position"]
    )

    stop["df_delta"] = df[
        ["RelLap", "Delta"]
    ]

    valid_stops.append(stop)

all_stops = valid_stops

print(
    f"Stops with valid pit position: {len(all_stops)}"
)

# --------------------------------------------------
# Aggregate by driver
# --------------------------------------------------
driver_sums = {}

for driver in sorted(
    set(s["Driver"] for s in all_stops)
):

    stops = [
        s for s in all_stops
        if s["Driver"] == driver
    ]

    if len(stops) < 2:
        continue

    drv_df = pd.concat(
        [s["df_delta"] for s in stops]
    )

    driver_sums[driver] = {
        "df": (
            drv_df.groupby("RelLap")["Delta"]
            .sum()
            .reset_index()
        ),

        "Color": driver_colors.get(
            driver,
            "#888888"
        ),

        "Count": len(stops)
    }

# --------------------------------------------------
# Combined dataframe
# --------------------------------------------------
all_deltas = pd.concat([
    s["df_delta"].assign(
        Driver=s["Driver"],
        Race=s["Race"],
        Color=s["Color"]
    )
    for s in all_stops
])

# --------------------------------------------------
# Figure
# --------------------------------------------------
fig = go.Figure()

# Background cloud
fig.add_trace(
    go.Scatter(
        x=all_deltas["RelLap"],
        y=all_deltas["Delta"],

        mode="markers",

        showlegend=False,

        marker=dict(
            color=all_deltas["Color"].tolist(),
            size=3,
            opacity=0.10
        ),

        customdata=all_deltas[
            ["Driver", "Race"]
        ].values,

        hovertemplate=(
            "<b>%{customdata[0]}</b>"
            " — %{customdata[1]}<br>"
            "Lap %{x:+d}"
            " · delta %{y:+d} pos"
            "<extra></extra>"
        )
    )
)

# Driver lines
for driver, data in driver_sums.items():

    fig.add_trace(
        go.Scatter(
            x=data["df"]["RelLap"],
            y=data["df"]["Delta"],

            mode="lines+markers",

            name=driver,

            line=dict(
                color=data["Color"],
                width=1.8
            ),

            marker=dict(
                size=5,
                color=data["Color"]
            ),

            opacity=0.85,

            hovertemplate=(
                f"<b>{driver}</b>"
                f" ({data['Count']} races)<br>"
                "Lap %{x:+d}"
                " · total %{y:+d} positions"
                "<extra></extra>"
            )
        )
    )

# --------------------------------------------------
# Reference lines
# --------------------------------------------------
fig.add_hline(
    y=0,

    line_color="rgba(255,255,255,0.15)",
    line_dash="dot",
    line_width=1
)

fig.add_vline(
    x=0,

    line_color="rgba(255,80,80,0.75)",
    line_dash="dash",
    line_width=1.5,

    annotation_text="PIT",

    annotation_font_color="#FF5050",
    annotation_font_size=12,

    annotation_position="top"
)

# --------------------------------------------------
# Layout
# --------------------------------------------------
fig.update_layout(

    title=dict(
        text=(
            "Cumulative Position Change Around "
            f"First Pit Stop — {YEAR} Season "
            f"({len(all_stops)} stops)"
        ),

        x=0.02,

        font=dict(
            size=22,
            color="#EAEAEA"
        )
    ),

    plot_bgcolor="#111111",
    paper_bgcolor="#111111",

    font=dict(
        color="#EAEAEA",
        family="Courier New"
    ),

    xaxis=dict(
        title="Laps Relative to Pit Stop",

        showgrid=True,

        gridcolor="rgba(255,255,255,0.06)",

        dtick=5,

        tickfont=dict(size=12),

        zeroline=False
    ),

    yaxis=dict(
        title="Total Positions Gained (+ = gained)",

        showgrid=True,

        gridcolor="rgba(255,255,255,0.06)",

        tickfont=dict(size=12),

        zeroline=False
    ),

    legend=dict(
        title="Driver",

        font=dict(size=11),

        bgcolor="rgba(20,20,20,0.8)",

        bordercolor="#444",
        borderwidth=1
    ),

    margin=dict(
        l=80,
        r=40,
        t=80,
        b=60
    ),

    height=660
)

fig.show()

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^

✗ Skipped Australian Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`


logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/_api.py", line 1730, in session_info
    raise SessionNotAvailableError(
fastf1._api.SessionNotAvailableError: No data for this session! If this session only finished recently, please try again in a few minutes.
req            INFO 	No cach

✗ Skipped Chinese Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`


logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/_api.py", line 1730, in session_info
    raise SessionNotAvailableError(
fastf1._api.SessionNotAvailableError: No data for this session! If this session only finished recently, please try again in a few minutes.
req            INFO 	No cach

...

logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/_api.py", line 1730, in session_info
    raise SessionNotAvailableError(
fastf1._api.SessionNotAvailableError: No data for this session! If this session only finished recently, please try again in a few minutes.
req            INFO 	No cach

✗ Skipped Qatar Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`


logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/_api.py", line 1730, in session_info
    raise SessionNotAvailableError(
fastf1._api.SessionNotAvailableError: No data for this session! If this session only finished recently, please try again in a few minutes.
req            INFO 	No cach

✗ Skipped Abu Dhabi Grand Prix: The data you are trying to access has not been loaded yet. See `Session.load`

Total pit stops collected: 0
Stops with valid pit position: 0


ValueError: No objects to concatenate

aaa

In [3]:
import os
import fastf1
import fastf1.plotting
import pandas as pd
import plotly.graph_objects as go

# --------------------------------------------------
# Cache
# --------------------------------------------------
os.makedirs("cache", exist_ok=True)
fastf1.Cache.enable_cache("cache")

YEAR = 2025

# --------------------------------------------------
# Load schedule
# --------------------------------------------------
schedule = fastf1.get_event_schedule(
    YEAR,
    include_testing=False
)

pit_rows = []
team_colors_map = {}

# --------------------------------------------------
# Collect pit lane times
# --------------------------------------------------
for _, event in schedule.iterrows():

    race_name = event["EventName"]
    round_num = event["RoundNumber"]

    try:
        session = fastf1.get_session(
            YEAR,
            round_num,
            "R"
        )

        session.load(
            laps=True,
            telemetry=False,
            weather=False,
            messages=False
        )

        print(f"✓ {race_name}")

    except Exception as e:
        print(f"✗ Skipped {race_name}: {e}")
        continue

    laps = session.laps.copy()

    # --------------------------------------------------
    # Team colors
    # --------------------------------------------------
    for _, row in session.results.iterrows():

        team = row["TeamName"]

        if team not in team_colors_map:

            try:
                team_colors_map[team] = (
                    fastf1.plotting.get_team_color(
                        team,
                        session=session
                    )
                )

            except Exception:
                team_colors_map[team] = "#888888"

    # --------------------------------------------------
    # Pit in / pit out pairing
    # --------------------------------------------------
    pit_in_laps = (
        laps[laps["PitInTime"].notna()][
            ["Driver", "LapNumber", "PitInTime"]
        ]
        .copy()
    )

    pit_out_laps = (
        laps[laps["PitOutTime"].notna()][
            ["Driver", "LapNumber", "PitOutTime"]
        ]
        .copy()
    )

    pit_in_laps["NextLap"] = (
        pit_in_laps["LapNumber"] + 1
    )

    merged = pit_in_laps.merge(
        pit_out_laps.rename(
            columns={"LapNumber": "NextLap"}
        ),
        on=["Driver", "NextLap"]
    )

    # --------------------------------------------------
    # Pit lane duration
    # --------------------------------------------------
    merged["Duration"] = (
        merged["PitOutTime"]
        - merged["PitInTime"]
    ).dt.total_seconds()

    # Remove corrupted values only
    merged = merged[
        (merged["Duration"] > 0)
        & (merged["Duration"] < 300)
    ]

    # --------------------------------------------------
    # Driver -> Team mapping
    # --------------------------------------------------
    driver_teams = (
        session.results
        .set_index("Abbreviation")["TeamName"]
        .to_dict()
    )

    merged["Team"] = (
        merged["Driver"].map(driver_teams)
    )

    merged = merged.dropna(subset=["Team"])

    pit_rows.append(
        merged[["Driver", "Team", "Duration"]]
    )

# --------------------------------------------------
# Final dataframe
# --------------------------------------------------
all_pits = pd.concat(
    pit_rows,
    ignore_index=True
)

print(
    f"\nTotal pit lane samples: {len(all_pits)}"
)

# --------------------------------------------------
# Sort by HIGHEST median pit lane time
# --------------------------------------------------
team_stats = (
    all_pits.groupby("Team")["Duration"]
    .agg(
        Median="median",
        Mean="mean",
        Count="count"
    )
    .sort_values(
        "Median",
        ascending=False
    )
)

teams_ordered = team_stats.index.tolist()

overall_median = (
    all_pits["Duration"].median()
)

# --------------------------------------------------
# Helper
# --------------------------------------------------
def shorten(name):

    return (
        name
        .replace(" Formula One Team", "")
        .replace(" F1 Team", "")
        .replace(" Racing", "")
        .replace(" Motorsport", "")
    )

# --------------------------------------------------
# Figure
# --------------------------------------------------
fig = go.Figure()

for team in teams_ordered:

    team_data = all_pits[
        all_pits["Team"] == team
    ]["Duration"]

    color = team_colors_map.get(
        team,
        "#888888"
    )

    # --------------------------------------------------
    # Convert HEX -> RGB
    # --------------------------------------------------
    hex_c = color.lstrip("#")

    try:
        r = int(hex_c[0:2], 16)
        g = int(hex_c[2:4], 16)
        b = int(hex_c[4:6], 16)

    except Exception:
        r, g, b = 140, 140, 140

    # --------------------------------------------------
    # Box plot
    # --------------------------------------------------
    fig.add_trace(
        go.Box(
            y=team_data,

            name=shorten(team),

            boxmean=False,

            quartilemethod="inclusive",

            width=0.48,

            whiskerwidth=0.5,

            line=dict(
                color=f"rgba({r},{g},{b},1)",
                width=2.6
            ),

            fillcolor=f"rgba({r},{g},{b},0.18)",

            medianline=dict(
                color="white",
                width=2.8
            ),

            boxpoints="suspectedoutliers",

            marker=dict(
                color=f"rgba({r},{g},{b},0.55)",
                size=4.5,
                opacity=0.35,
                line=dict(width=0)
            ),

            hovertemplate=(
                "<b>%{x}</b><br>"
                "Pit lane time: %{y:.2f}s"
                "<extra></extra>"
            ),
        )
    )

# --------------------------------------------------
# Reference line
# --------------------------------------------------
fig.add_hline(
    y=overall_median,

    line=dict(
        color="rgba(255,255,255,0.20)",
        dash="dot",
        width=1.4
    ),

    annotation_text=(
        f"Season median  "
        f"{overall_median:.1f}s"
    ),

    annotation_position="top right",

    annotation_font=dict(
        size=11,
        color="rgba(210,210,210,0.55)"
    ),
)

# --------------------------------------------------
# Median labels
# --------------------------------------------------
annotations = []

for team in teams_ordered:

    median_val = (
        team_stats.loc[team, "Median"]
    )

    count = int(
        team_stats.loc[team, "Count"]
    )

    q3 = (
        all_pits[
            all_pits["Team"] == team
        ]["Duration"]
        .quantile(0.75)
    )

    annotations.append(
        dict(
            x=shorten(team),
            y=q3,

            yshift=18,

            text=(
                f"<b>{median_val:.1f}s</b>"
            ),

            showarrow=False,

            font=dict(
                size=11,
                color="rgba(240,240,240,0.92)"
            ),
        )
    )

    annotations.append(
        dict(
            x=shorten(team),
            y=0,
            yref="paper",

            yshift=-30,

            text=f"n={count}",

            showarrow=False,

            font=dict(
                size=9,
                color="rgba(170,170,170,0.45)"
            ),
        )
    )

# --------------------------------------------------
# Layout styling
# --------------------------------------------------
fig.update_layout(

    title=dict(
        text=(
            f"<b>F1 Team Pit Lane Time</b>"
            f"<br><sup>{YEAR} Season</sup>"
        ),

        x=0.03,

        font=dict(
            size=30,
            color="#F5F5F5"
        ),
    ),

    annotations=annotations,

    plot_bgcolor="#0B0B0B",
    paper_bgcolor="#0B0B0B",

    font=dict(
        family="Inter, Arial",
        color="#EAEAEA"
    ),

    xaxis=dict(
        title=None,

        showgrid=False,

        tickangle=-32,

        tickfont=dict(
            size=12,
            color="#D9D9D9"
        ),

        zeroline=False,
    ),

    yaxis=dict(
        title=dict(
            text="Pit Lane Time (seconds)",

            font=dict(
                size=16
            )
        ),

        showgrid=True,

        gridcolor="rgba(255,255,255,0.05)",

        griddash="dot",

        tickfont=dict(
            size=12
        ),

        zeroline=False,
    ),

    margin=dict(
        l=85,
        r=40,
        t=95,
        b=150
    ),

    height=720,

    showlegend=False,
)

fig.show()

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
DEBUG:fastf1.api:Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
DEBUG:fastf1.fastf1.core:Traceback for failure in session info data
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ^^^^^^^^^^

✓ Australian Grand Prix


DataNotLoadedError: The data you are trying to access has not been loaded yet. See `Session.load`

aaa

---


<h1 align="center"><b>Graph 3: Aerodynamic Performance and Downforce</b></h1>



---

In [ ]:
# Graph Code

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

---


<h1 align="center"><b>Graph 4: DRS Speed Gains</b></h1>



---

In [ ]:
# Graph Code

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.

---


<h1 align="center"><b>Graph 5: Car Performances in Corners and Straights</b></h1>



---

In [ ]:
# Graph Code

Lorem Ipsum is simply dummy text of the printing and typesetting industry. Lorem Ipsum has been the industry's standard dummy text ever since the 1500s, when an unknown printer took a galley of type and scrambled it to make a type specimen book. It has survived not only five centuries, but also the leap into electronic typesetting, remaining essentially unchanged. It was popularised in the 1960s with the release of Letraset sheets containing Lorem Ipsum passages, and more recently with desktop publishing software like Aldus PageMaker including versions of Lorem Ipsum.